# Urdu-English NMT — Data Pipeline
### Notebook 01

Runs the full data download + cleaning pipeline on Kaggle and produces the train/val/test TSV splits used by all subsequent notebooks.

**Output files (in `/kaggle/working/data/processed/`):**
- `train.tsv` — ~450k pairs for training
- `val.tsv` — ~5k pairs for validation  
- `test.tsv` — ~10k pairs for evaluation
- `urdu_train_only.txt` — Urdu column only (for SentencePiece training)

In [ ]:
# ============================================================
# CELL 1: Install dependencies
# ============================================================

import subprocess, sys

packages = ['tqdm', 'langdetect', 'sacrebleu', 'transformers', 'sentencepiece']
for pkg in packages:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], capture_output=True)

print('Dependencies ready.')

In [ ]:
# ============================================================
# CELL 2: Clone repo and configure paths
# ============================================================

import os, sys

REPO_URL     = 'https://github.com/YOUR_USERNAME/urdu-en-nmt.git'  # <-- CHANGE
PROJECT_ROOT = '/kaggle/working/urdu-en-nmt'

# Clone repo if not already present
if not os.path.exists(PROJECT_ROOT):
    os.system(f'git clone {REPO_URL} {PROJECT_ROOT}')
    print(f'Cloned repo to {PROJECT_ROOT}')
else:
    print(f'Repo already exists at {PROJECT_ROOT}')

# Add to Python path
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Set environment variables for data/results directories
os.environ['NMT_DATA_DIR']    = '/kaggle/working/data'
os.environ['NMT_RESULTS_DIR'] = '/kaggle/working/results'

os.makedirs('/kaggle/working/data/raw',       exist_ok=True)
os.makedirs('/kaggle/working/data/processed', exist_ok=True)
os.makedirs('/kaggle/working/results',        exist_ok=True)

print('Environment configured.')
print(f'  DATA_DIR   : {os.environ["NMT_DATA_DIR"]}')
print(f'  RESULTS_DIR: {os.environ["NMT_RESULTS_DIR"]}')

In [ ]:
# ============================================================
# CELL 3: Run the data pipeline
# Calls download_and_clean.py which:
#   1. Downloads raw OPUS ZIP files (TED2020, GNOME, Tanzil)
#   2. Parses Moses-format sentence pairs
#   3. Applies all cleaning filters (script, length, ratio, content, lang)
#   4. Deduplicates and removes cross-split overlap
#   5. Splits into train/val/test TSV files
#
# Expected runtime: 15–25 minutes (dominated by download time and
# langdetect on ~800k pairs)
# ============================================================

import subprocess

result = subprocess.run(
    ['python', os.path.join(PROJECT_ROOT, 'data', 'download_and_clean.py'),
     '--output_dir', '/kaggle/working/data'],
    capture_output=False,   # Print output in real time
    text=True,
)

print(f'\nReturn code: {result.returncode}')
if result.returncode != 0:
    print('Pipeline failed. Check output above for errors.')
else:
    print('Pipeline complete!')

In [ ]:
# ============================================================
# CELL 4: Verify output files
# Print split sizes and a few random samples to confirm
# the pipeline ran correctly.
# ============================================================

processed_dir = '/kaggle/working/data/processed'

print('Verifying output files...')
print(f'{"File":<35} {"Lines":>12}')
print('-' * 50)

for filename in ['train.tsv', 'val.tsv', 'test.tsv', 'urdu_train_only.txt']:
    path = os.path.join(processed_dir, filename)
    if os.path.exists(path):
        n = sum(1 for _ in open(path, encoding='utf-8'))
        size_mb = os.path.getsize(path) / 1e6
        print(f'{filename:<35} {n:>12,}  ({size_mb:.1f} MB)')
    else:
        print(f'{filename:<35} NOT FOUND')

# Print 3 random samples from train.tsv
import random
train_path = os.path.join(processed_dir, 'train.tsv')
if os.path.exists(train_path):
    all_lines = open(train_path, encoding='utf-8').readlines()
    samples = random.Random(42).sample(all_lines, 3)
    print('\n3 Random Training Samples:')
    for i, line in enumerate(samples, 1):
        ur, en = line.strip().split('\t', 1)
        print(f'  [{i}] Urdu   : {ur}')
        print(f'       English: {en}')
        print()